In [1]:
# Installation des dépendances
!pip install requests openpyxl sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 556.9 kB/s eta 0:00:00a 0:00:01


In [3]:
print(pd.__version__)

2.2.3


In [8]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from io import BytesIO

# Configuration PostgreSQL
DB_USER = "admin"
DB_PASSWORD = "motdepasse_secret"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "greenandcoop"

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# URLs des fichiers Excel
sources = {
    "WeatherUnderground_LaMadeleine_FR_File": "https://s3.eu-west-1.amazonaws.com/course.oc-static.com/projects/922_Data+Engineer/922_P8/Weather+Underground+-+La+Madeleine%2C+FR.xlsx",
    "WeatherUnderground_Ichtegem_BE_File": "https://s3.eu-west-1.amazonaws.com/course.oc-static.com/projects/922_Data+Engineer/922_P8/Weather+Underground+-+Ichtegem%2C+BE.xlsx"
}

for table_name, url in sources.items():
    print(f"\nTraitement de {table_name}...")
    
    # Téléchargement du fichier
    response = requests.get(url)
    excel_file = BytesIO(response.content)
    
    # Lecture de toutes les feuilles
    all_sheets = pd.read_excel(excel_file, sheet_name=None, engine='openpyxl')
    
    dfs = []
    for sheet_name, df in all_sheets.items():
        
        if sheet_name == list(all_sheets.keys())[0]:
            print(f"Colonnes : {df.columns.tolist()}")
        
        date_str = sheet_name
        date = pd.to_datetime(date_str, format='%d%m%y')
        
        # Construire le timestamp complet
        df['date'] = date
        df = df.dropna(subset=['Time']).copy()
        df['measured_at'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['Time'].astype(str),format='%Y-%m-%d %H:%M:%S')
        df = df.drop(columns=['date'])
        
        dfs.append(df)
        print(f"  Feuillet {sheet_name} : {len(df)} lignes")
    
    df_final = pd.concat(dfs, ignore_index=True)
    print(f"  Total : {len(df_final)} lignes")
    
    # Supprimer la table avec CASCADE
    with engine.connect() as conn:
        conn.execute(text(f'DROP TABLE IF EXISTS raw."{table_name}" CASCADE'))
        conn.commit()
    
    # Charger dans PostgreSQL
    df_final.to_sql(
        name=table_name,
        schema='raw',
        con=engine,
        if_exists='replace',
        index=False
    )
    print(f"  ✅ Table raw.{table_name} mise à jour avec dates complètes !")

print("\nTerminé !")


Traitement de WeatherUnderground_LaMadeleine_FR_File...
Colonnes : ['Time', 'Temperature', 'Dew Point', 'Humidity', 'Wind', 'Speed', 'Gust', 'Pressure', 'Precip. Rate.', 'Precip. Accum.', 'UV', 'Solar']
  Feuillet 011024 : 288 lignes
  Feuillet 021024 : 288 lignes
  Feuillet 031024 : 288 lignes
  Feuillet 041024 : 288 lignes
  Feuillet 051024 : 288 lignes
  Feuillet 061024 : 288 lignes
  Feuillet 071024 : 180 lignes
  Total : 1908 lignes
  ✅ Table raw.WeatherUnderground_LaMadeleine_FR_File mise à jour avec dates complètes !

Traitement de WeatherUnderground_Ichtegem_BE_File...
Colonnes : ['Time', 'Temperature', 'Dew Point', 'Humidity', 'Wind', 'Speed', 'Gust', 'Pressure', 'Precip. Rate.', 'Precip. Accum.', 'UV', 'Solar']
  Feuillet 011024 : 288 lignes
  Feuillet 021024 : 285 lignes
  Feuillet 031024 : 284 lignes
  Feuillet 041024 : 288 lignes
  Feuillet 051024 : 288 lignes
  Feuillet 061024 : 288 lignes
  Feuillet 071024 : 178 lignes
  Total : 1899 lignes
  ✅ Table raw.WeatherUndergro

In [9]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from io import BytesIO

# Configuration PostgreSQL RDS
DB_USER = "axel"
DB_PASSWORD = "motdepasse_secret"
DB_HOST = "greenandcoop-db.cxmusuii8lhc.eu-west-3.rds.amazonaws.com"
DB_PORT = "5432"
DB_NAME = "greenandcoop"

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require")

# URLs des fichiers Excel
sources = {
    "WeatherUnderground_LaMadeleine_FR_File": "https://s3.eu-west-1.amazonaws.com/course.oc-static.com/projects/922_Data+Engineer/922_P8/Weather+Underground+-+La+Madeleine%2C+FR.xlsx",
    "WeatherUnderground_Ichtegem_BE_File": "https://s3.eu-west-1.amazonaws.com/course.oc-static.com/projects/922_Data+Engineer/922_P8/Weather+Underground+-+Ichtegem%2C+BE.xlsx"
}

for table_name, url in sources.items():
    print(f"\nTraitement de {table_name}...")
    
    # Téléchargement du fichier
    response = requests.get(url)
    excel_file = BytesIO(response.content)
    
    # Lecture de toutes les feuilles
    all_sheets = pd.read_excel(excel_file, sheet_name=None, engine='openpyxl')
    
    dfs = []
    for sheet_name, df in all_sheets.items():
        
        if sheet_name == list(all_sheets.keys())[0]:
            print(f"Colonnes : {df.columns.tolist()}")
        
        date_str = sheet_name
        date = pd.to_datetime(date_str, format='%d%m%y')
        
        # Construire le timestamp complet
        df['date'] = date
        df = df.dropna(subset=['Time']).copy()
        df['measured_at'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['Time'].astype(str),format='%Y-%m-%d %H:%M:%S')
        df = df.drop(columns=['date'])
        
        dfs.append(df)
        print(f"  Feuillet {sheet_name} : {len(df)} lignes")
    
    df_final = pd.concat(dfs, ignore_index=True)
    print(f"  Total : {len(df_final)} lignes")
    
    # Supprimer la table avec CASCADE
    with engine.connect() as conn:
        conn.execute(text(f'DROP TABLE IF EXISTS raw."{table_name}" CASCADE'))
        conn.commit()
    
    # Charger dans PostgreSQL
    df_final.to_sql(
        name=table_name,
        schema='raw',
        con=engine,
        if_exists='replace',
        index=False
    )
    print(f"  ✅ Table raw.{table_name} mise à jour avec dates complètes !")

print("\nTerminé !")


Traitement de WeatherUnderground_LaMadeleine_FR_File...
Colonnes : ['Time', 'Temperature', 'Dew Point', 'Humidity', 'Wind', 'Speed', 'Gust', 'Pressure', 'Precip. Rate.', 'Precip. Accum.', 'UV', 'Solar']
  Feuillet 011024 : 288 lignes
  Feuillet 021024 : 288 lignes
  Feuillet 031024 : 288 lignes
  Feuillet 041024 : 288 lignes
  Feuillet 051024 : 288 lignes
  Feuillet 061024 : 288 lignes
  Feuillet 071024 : 180 lignes
  Total : 1908 lignes
  ✅ Table raw.WeatherUnderground_LaMadeleine_FR_File mise à jour avec dates complètes !

Traitement de WeatherUnderground_Ichtegem_BE_File...
Colonnes : ['Time', 'Temperature', 'Dew Point', 'Humidity', 'Wind', 'Speed', 'Gust', 'Pressure', 'Precip. Rate.', 'Precip. Accum.', 'UV', 'Solar']
  Feuillet 011024 : 288 lignes
  Feuillet 021024 : 285 lignes
  Feuillet 031024 : 284 lignes
  Feuillet 041024 : 288 lignes
  Feuillet 051024 : 288 lignes
  Feuillet 061024 : 288 lignes
  Feuillet 071024 : 178 lignes
  Total : 1899 lignes
  ✅ Table raw.WeatherUndergro